In [ ]:
pip install cooper-optim

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import cooper
from cooper.penalty_coefficients import MultiplicativePenaltyCoefficientUpdater

In [ ]:
def compute_d(arch):
    d = 0
    for i in range(len(arch) - 1):
        d += arch[i] * arch[i + 1]
    return d

In [ ]:
def unpack_weights(w_flat, arch):
    matrices = []
    idx = 0
    for i in range(len(arch) - 1):
        n_in, n_out = arch[i], arch[i + 1]
        size = n_in * n_out
        W = w_flat[idx: idx + size].reshape(n_out, n_in)
        matrices.append(W)
        idx += size
    return matrices

In [ ]:
def forward_pass(u, w_flat, biases, arch):
    weight_mats = unpack_weights(w_flat, arch)
    a = u
    for l, W in enumerate(weight_mats):
        z = W @ a + biases[l].unsqueeze(1)
        if l < len(weight_mats) - 1:
            a = torch.tanh(z)
        else:
            a = z
    return a

In [ ]:
def binary_entropy(Q, eps=1e-6):
    Qc = Q.clamp(eps, 1 - eps) # 
    return -(Qc * Qc.log() + (1 - Qc) * (1 - Qc).log()).sum()

In [ ]:
class SparseNNModel(nn.Module):
    def __init__(self, arch, k, eps=1e-6):
        super().__init__()
        self.arch = arch
        self.k = k
        self.eps = eps
        self.d = compute_d(arch)

        # Q initialized column-stochastic
        Q_init = torch.ones(self.d, k) / self.d
        self.Q = nn.Parameter(Q_init)

        # reduced coefficients
        self.x_coeff = nn.Parameter(torch.randn(k) * 0.1)

        # biases
        self.biases = nn.ParameterList()
        for i in range(len(arch) - 1):
            self.biases.append(nn.Parameter(torch.zeros(arch[i + 1])))

    def get_w(self):
        return self.Q @ self.x_coeff

    def forward(self, u):
        return forward_pass(u, self.get_w(), self.biases, self.arch)

    def clamp_Q(self):
        with torch.no_grad():
            self.Q.clamp_(self.eps, 1.0 - self.eps) # 

In [ ]:
Q = torch.ones(120,20) / 120
x = torch.randn(20) * 0.1
w = Q @ x
print(w)

In [ ]:
class SparseNNCMP(cooper.ConstrainedMinimizationProblem):
    def __init__(
        self,
        model,
        c0,
        u_data,
        y_data,
        ent_tol=1e-1, # ...
        col_tol=1e-3, # ...
        init_penalty_entropy=1.0, # hyperparameter for entropy penalty
        init_penalty_colsum=5.0, # hyperparameter for column sum penalty
        tol=1e-6, 
    ):
        super().__init__()
        self.model = model
        self.c0 = c0
        self.u_data = u_data
        self.y_data = y_data
        self.penalty_entropy = init_penalty_entropy
        self.penalty_colsum = init_penalty_colsum
        self.tol = tol

        # H(Q) - c0 <= 0
        self.entropy_eq = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=1),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.tensor([init_penalty_entropy], dtype=torch.float32)
            ),
        )

        # Q^T 1_d - 1_k - tol <= 0
        self.colsum_eq_upper = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=model.k),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.full((model.k,), float(init_penalty_colsum), dtype=torch.float32)
            ),
        )

        # -Q^T 1_d + 1_k - tol <= 0
        self.colsum_eq_lower = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=model.k),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.full((model.k,), float(init_penalty_colsum), dtype=torch.float32)
            ),
        )


    def compute_cmp_state(self):
        m = self.u_data.shape[1]
        y_pred = self.model(self.u_data)
        loss = ((self.y_data - y_pred) ** 2).sum() / m

        H = binary_entropy(self.model.Q, eps=self.model.eps)
        entropy_violation = (H - self.c0).reshape(1) # (1,)

        colsum_violation_upper = self.model.Q.sum(dim=0) - 1.0 - self.tol # (20,)
        colsum_violation_lower = 1.0 - self.model.Q.sum(dim=0) - self.tol # (20,)

        return cooper.CMPState(
            loss=loss,
            observed_constraints={
                self.entropy_eq: cooper.ConstraintState(violation=entropy_violation),
                self.colsum_eq_upper: cooper.ConstraintState(violation=colsum_violation_upper),
                self.colsum_eq_lower: cooper.ConstraintState(violation=colsum_violation_lower),
            },
        )

    # Debug function to print column gradient components
    def print_column_gradients(self, rows=[0, 1, 2], cols=[0, 1, 2]):
        model = self.model
        Q = model.Q
        x = model.x_coeff

        # forward loss via explicit w so we can differentiate wrt w later
        m = self.u_data.shape[1]
        w = model.get_w()
        y_pred = forward_pass(self.u_data, w, model.biases, model.arch)
        loss = ((self.y_data - y_pred) ** 2).sum() / m

        # entropy and constraints
        H = binary_entropy(Q, eps=model.eps)
        colsum = Q.sum(dim=0)
        col_violation_upper = colsum - 1.0              # shape (k,)
        col_violation_lower = 1.0 - colsum              # shape (k,)

        # individual gradients
        grad_loss = torch.autograd.grad(loss, Q, retain_graph=True)[0]
        grad_H = torch.autograd.grad(H, Q, retain_graph=True)[0]

        grad_col_up = torch.autograd.grad(col_violation_upper.sum(), Q, retain_graph=True)[0]
        grad_col_lo = torch.autograd.grad(col_violation_lower.sum(), Q, retain_graph=True)[0]

        # print selected entries
        print("\n--- Gradient entries for selected Q[i,j] ---")
        for i in rows:
            for j in cols:
                task = grad_loss[i, j].item()
                ent = grad_H[i, j].item()
                up = grad_col_up[i, j].item()
                lo = grad_col_lo[i, j].item()

                print(
                    f"Q[{i},{j}] | "
                    f"task={task:.4e}, "
                    f"entropy={ent:.4e}, "
                    f"col_up={up:.4e}, "
                    f"col_lo={lo:.4e}"
                )

        # verify task gradient formula g_i * x_j using the same w as above
        g_w = torch.autograd.grad(loss, w, retain_graph=True)[0]

        print("\n--- Check task gradient = g_i * x_j ---")
        for i in rows:
            for j in cols:
                formula_val = (g_w[i] * x[j]).item()
                autodiff_val = grad_loss[i, j].item()
                print(
                    f"Q[{i},{j}] | autodiff={autodiff_val:.4e}, formula={formula_val:.4e}"
                )

        # useful summaries
        print("\n--- Norms ---")
        print("||grad_loss||   =", grad_loss.norm().item())
        print("||grad_entropy||=", grad_H.norm().item())
        print("||grad_col_up|| =", grad_col_up.norm().item())
        print("||grad_col_lo|| =", grad_col_lo.norm().item())

    def print_selected_gradients(self, rows=(50, 56, 111, 114), cols=(0, 1, 2, 3)):
        model = self.model
        Q = model.Q
        x = model.x_coeff
        m = self.u_data.shape[1]

        w = model.get_w()
        y_pred = forward_pass(self.u_data, w, model.biases, model.arch)
        loss = ((self.y_data - y_pred) ** 2).sum() / m

        H = binary_entropy(Q, eps=model.eps)
        colsum = Q.sum(dim=0)

        ent = H - self.c0 
        col_up = colsum - 1.0 - self.col_tol
        col_lo = 1.0 - colsum - self.col_tol

        grad_loss_Q = torch.autograd.grad(loss, Q, retain_graph=True)[0]
        grad_H_Q = torch.autograd.grad(H, Q, retain_graph=True)[0]

        lam_E = self.entropy_eq.multiplier.weight
        lam_U = self.colsum_upper.multiplier.weight
        lam_L = self.colsum_lower.multiplier.weight

        rho_E = self.entropy_eq.penalty_coefficient.value
        rho_U = self.colsum_upper.penalty_coefficient.value
        rho_L = self.colsum_lower.penalty_coefficient.value

        ent_term = (lam_E * ent).sum() + 0.5 * (rho_E * torch.relu(ent) ** 2).sum()
        col_up_term = (lam_U * col_up).sum() + 0.5 * (rho_U * torch.relu(col_up) ** 2).sum()
        col_lo_term = (lam_L * col_lo).sum() + 0.5 * (rho_L * torch.relu(col_lo) ** 2).sum()

        grad_ent_Q = torch.autograd.grad(ent_term, Q, retain_graph=True)[0]
        grad_col_up_Q = torch.autograd.grad(col_up_term, Q, retain_graph=True)[0]
        grad_col_lo_Q = torch.autograd.grad(col_lo_term, Q, retain_graph=True)[0]

        g_w = torch.autograd.grad(loss, w, retain_graph=True)[0]

        print("\n--- Selected Q[i,j] gradients ---")
        for i in rows:
            for j in cols:
                print(
                    f"Q[{i},{j}] | "
                    f"task={grad_loss_Q[i,j].item(): .3e}  "
                    f"Hraw={grad_H_Q[i,j].item(): .3e}  "
                    f"E_AL={grad_ent_Q[i,j].item(): .3e}  "
                    f"U_AL={grad_col_up_Q[i,j].item(): .3e}  "
                    f"L_AL={grad_col_lo_Q[i,j].item(): .3e}"
                )

        print("\n--- Check task gradient formula dL/dQ_ij = (dL/dw_i) x_j ---")
        for i in rows:
            for j in cols:
                print(
                    f"Q[{i},{j}] | autodiff={grad_loss_Q[i,j].item(): .3e}  "
                    f"formula={(g_w[i] * x[j]).item(): .3e}"
                )

        print("\n--- Gradient norms ---")
        print("||grad_loss_Q||   =", grad_loss_Q.norm().item())
        print("||grad_H_Q||      =", grad_H_Q.norm().item())
        print("||grad_ent_Q||    =", grad_ent_Q.norm().item())
        print("||grad_col_up_Q|| =", grad_col_up_Q.norm().item())
        print("||grad_col_lo_Q|| =", grad_col_lo_Q.norm().item())



In [ ]:
# Homotopy Training
def run_homotopy(
    arch,
    u_data,
    y_data,
    k,
    c0_start,
    beta=0.97,
    outer_iters=105,
    inner_iters=300,
    ent_tol=1e-1,
    col_tol=1e-3,
    primal_lr=1e-3, # learning rate
    dual_lr=1e-4, # learning rate 
    eps=1e-6,
    penalty_growth=1.2,
    init_penalty_entropy=1.0,
    init_penalty_colsum=5.0,
):
    torch.manual_seed(42)

    d = compute_d(arch)
    print(f"Architecture: {arch}, d={d}, k={k}")
    print(f"c0_start={c0_start:.4f}, beta={beta}, outer={outer_iters}, inner={inner_iters}")

    model = SparseNNModel(arch, k, eps=eps)
    print(f"Initial entropy H(Q) = {binary_entropy(model.Q, eps).item():.4f}")
    
    # Build CMP
    cmp = SparseNNCMP(
        model=model,
        c0=c0_start,
        u_data=u_data,
        y_data=y_data,
        ent_tol=ent_tol,
        col_tol=col_tol,
        init_penalty_entropy=init_penalty_entropy,
        init_penalty_colsum=init_penalty_colsum,
        tol=eps, # ...

    )
    
    primal_opt = torch.optim.Adam(model.parameters(), lr=primal_lr)
    dual_opt = torch.optim.SGD(cmp.dual_parameters(), lr=dual_lr, maximize=True) # ...

    cooper_opt = cooper.optim.AlternatingPrimalDualOptimizer(
        cmp=cmp,
        primal_optimizers=primal_opt,
        dual_optimizers=dual_opt,
    )
    entropy_updater = MultiplicativePenaltyCoefficientUpdater(
        growth_factor=penalty_growth,
        violation_tolerance=ent_tol,
        has_restart=True,
        )
    colsum_updater = MultiplicativePenaltyCoefficientUpdater(
        growth_factor=penalty_growth,
        violation_tolerance=col_tol,
        has_restart=False,
    )
    history = []
    c0 = c0_start

    for outer in range(outer_iters):
        cmp.c0 = c0

        for _ in range(inner_iters):
            cooper_opt.roll()
        
        model.clamp_Q() # ...
            
        # column-sum multipliers update
        
        with torch.no_grad(): # memory efficient to compute these values without tracking gradients
            Q_now = model.Q.detach().clone()
            H_val = binary_entropy(Q_now, eps).item()
            colsum_err = (Q_now.sum(dim=0) - 1.0).abs().max().item()
            y_pred = model(u_data)
            loss_val = ((y_data - y_pred) ** 2).sum().item() / u_data.shape[1]
            gap = torch.minimum(Q_now, 1.0 - Q_now).max().item()

            cE = cmp.entropy_eq.penalty_coefficient.value[0].item()
            cCmax = cmp.colsum_eq_upper.penalty_coefficient.value.max().item()

        history.append(
            {
                "outer": outer,
                "c0": c0,
                "H": H_val,
                "loss": loss_val,
                "gap": gap,
                "colsum_err": colsum_err,
                "cE": cE,
                "cCmax": cCmax,
            }
        )

        # print column gradients ...
        print(f"\n=== After outer iteration {outer} ===")
        print(f"c0 = {c0:.4f}, H(Q) = {H_val:.4f}, loss = {loss_val:.4e}, gap = {gap:.4e}, colsum_err = {colsum_err:.4e}")
        print(f"Penalty coefficients: cE = {cE:.4f}, cCmax = {cCmax:.4f}")
        # cmp.print_column_gradients(rows=[0, 1, 2], cols=[0, 1, 2])

        # print selected gradients
        print(f"\nSelected gradients for outer iteration {outer}:")
        cmp.print_selected_gradients(rows=(50, 56, 111, 114), cols=(0, 1, 2, 3))

        c0 *= beta

    return model, history

In [ ]:
def main():
    np.random.seed(42)

    # Data
    m = 500
    u_np = np.random.uniform(-2, 2, size=(1, m))
    y_np = u_np**3 + u_np**2 + 1.0 + 0.1 * np.random.randn(1, m)

    u_data = torch.tensor(u_np, dtype=torch.float32)
    y_data = torch.tensor(y_np, dtype=torch.float32)

    # Architecture
    arch = [1, 10, 10, 1]
    k = 20
    d = compute_d(arch)
    print(f"d = {d} (should be 120)")

    tmp_model = SparseNNModel(arch, k)
    c0_start = 0.99 * binary_entropy(tmp_model.Q).item()

    del tmp_model # free memory before main optimization
    print(f"Starting c0 = {c0_start:.4f}")

    model, history = run_homotopy(
        arch=arch,
        u_data=u_data,
        y_data=y_data,
        k=k,
        c0_start=c0_start,
        beta=0.97, # ...
        outer_iters=105,
        inner_iters=300,
        primal_lr=1e-3, # ...
        dual_lr=1e-4, # ...
        eps=1e-6, # ...
        ent_tol=1e-1,
        col_tol=1e-3,
        penalty_growth=1.2,
        init_penalty_entropy=1.0, # ...
        init_penalty_colsum=5.0, # ...
    )
    Q_final = model.Q.detach().cpu().numpy()
    x_final = model.x_coeff.detach().cpu().numpy()
    w_final = Q_final @ x_final

    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)

    if len(history) > 0:
        print(f"Final Q binary gap: {np.max(np.minimum(Q_final, 1.0 - Q_final)):.6f}")
        print(f"Final colsum error: {np.max(np.abs(Q_final.sum(axis=0) - 1.0)):.6f}")
        print(f"Final entropy H(Q): {binary_entropy(model.Q).item():.6f}")
        print(f"Final loss: {history[-1]['loss']:.6f}")

    print("\nEntries of Q > 0.9:")
    rows, cols = np.where(Q_final > 0.9)
    for r, c in zip(rows, cols):
        print(f"  Q[{r},{c}] = {Q_final[r, c]:.6f}")




if __name__ == "__main__":
    main()